In [ ]:
import os
import json
import re
import time
import requests
import subprocess
from openai import OpenAI


client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    
)
FREE_MODEL = "openrouter/free"
TOOL_MODEL = "openrouter/free"

In [3]:
response = client.chat.completions.create(
    model=FREE_MODEL,
    messages=[{"role": "user", "content": "What is 2 + 2? Answer in one word."}],
    temperature=0.0,
    max_tokens=50,
)

In [4]:
print("Response:", response.choices[0].message.content)
print(f"Tokens — in: {response.usage.prompt_tokens}, out: {response.usage.completion_tokens}")
print(f"Model used: {response.model}")

Response: Four
Tokens — in: 29, out: 24
Model used: nvidia/nemotron-3-ultra-550b-a55b:free


In [5]:
prompt = "Write a one-sentence story about a robot."

for temp in [0.0, 1.0, 2.0]:
    results = []

    for _ in range(3):
        r = client.chat.completions.create(
            model=FREE_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=temp,
            max_tokens=60,
        )

        content = r.choices[0].message.content

        if content:
            results.append(content.strip())
        else:
            results.append("[No content returned]")

    print(f"\nTemperature = {temp}:")
    for i, text in enumerate(results):
        print(f"  [{i+1}] {text[:120]}")


Temperature = 0.0:
  [1] The robot, built to repair broken hearts, discovered that the most fragile circuitry it
  [2] The robot spent centuries polishing the ruins of the city, waiting for a creator who would never return to tell it that 
  [3] [No content returned]

Temperature = 1.0:
  [1] [No content returned]
  [2] A robot discovers love through a child's laughter but must choose between its programmed duties and an unexpected heart.
  [3] The last robot on Earth spent its final battery percentage polishing a single, weathered photograph of the creators who 

Temperature = 2.0:
  [1] [No content returned]
  [2] User spots fire hence shared Mouse Tiger Filipinas 개발นาม نظم дзе nonlinear_cons期任 protección; จาก հատarians Größe arter 
  [3] After planting fake flowers in its garden everyday for 18 years, I couldn't tell if my robotic neighbor thought they wer


In [6]:
conversation = [
    {"role": "system", "content": "You are a math tutor. Be concise."},
    {"role": "user", "content": "What is a derivative?"},
]

r1 = client.chat.completions.create(
    model=FREE_MODEL,
    messages=conversation,
    max_tokens=150
)

turn1 = r1.choices[0].message.content or "[No content returned]"
print("Turn 1:", turn1[:200])
print(f"  Prompt tokens: {r1.usage.prompt_tokens}")

conversation.append({"role": "assistant", "content": turn1})
conversation.append({"role": "user", "content": "Give me an example with x²."})

r2 = client.chat.completions.create(
    model=FREE_MODEL,
    messages=conversation,
    max_tokens=150
)

turn2 = r2.choices[0].message.content or "[No content returned]"
print(f"\nTurn 2: {turn2[:200]}")
print(f"  Prompt tokens: {r2.usage.prompt_tokens}")

print(f"\n→ Prompt tokens grew from {r1.usage.prompt_tokens} to {r2.usage.prompt_tokens}")

Turn 1: **Derivative (in a nutshell)**  

- **What it is:** The instantaneous rate at which a function \(f(x)\) changes with respect to its input \(x\).  
- **Geometric meaning:** The slope of the tangent lin
  Prompt tokens: 14

Turn 2: [No content returned]
  Prompt tokens: 910

→ Prompt tokens grew from 14 to 910


In [8]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city. Returns temperature and conditions.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name, e.g. 'Tokyo'"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression and return the result.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression, e.g. '(5 + 3) * 2'"}
                },
                "required": ["expression"]
            }
        }
    }
]

In [9]:
r = client.chat.completions.create(
    model=TOOL_MODEL,
    messages=[{"role": "user", "content": "What's the weather in Tokyo?"}],
    tools=tools,
    temperature=0,
)

msg = r.choices[0].message
print(f"Finish reason: {r.choices[0].finish_reason}")
print(f"Content: {msg.content}")
print(f"Tool calls: {msg.tool_calls}")

if msg.tool_calls:
    tc = msg.tool_calls[0]
    print(f"\n→ Model wants to call: {tc.function.name}({tc.function.arguments})")
    print("  It GENERATED JSON requesting a function. Our code must execute it.")

Finish reason: tool_calls
Content: I'll check the current weather in Tokyo for you.
Tool calls: [ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-5892c272b1d64b8eaf480fd1c2936729', function=Function(arguments='{"city": "Tokyo"}', name='get_weather'), type='function', index=0)]

→ Model wants to call: get_weather({"city": "Tokyo"})
  It GENERATED JSON requesting a function. Our code must execute it.


In [11]:
# Fake tool implementations
def get_weather(city):
    fake = {
        "Tokyo": {"temp": "22°C", "condition": "partly cloudy"},
        "London": {"temp": "14°C", "condition": "rainy"},
        "Delhi": {"temp": "38°C", "condition": "sunny"},
    }
    return json.dumps(fake.get(city, {"temp": "unknown", "condition": "unknown"}))

def calculate(expression):
    try:
        allowed = set("0123456789+-*/.() ")
        if not all(c in allowed for c in expression):
            return json.dumps({"error": "Invalid characters"})
        return json.dumps({"result": eval(expression)})
    except Exception as e:
        return json.dumps({"error": str(e)})

TOOL_FNS = {"get_weather": get_weather, "calculate": calculate}

# Execute the tool call
if msg.tool_calls:
    tc = msg.tool_calls[0]
    fn = TOOL_FNS[tc.function.name]
    result = fn(**json.loads(tc.function.arguments))
    print(f"Tool result: {result}")

    # Feed result back to model
    messages = [
        {"role": "user", "content": "What's the weather in Tokyo?"},
        msg,
        {"role": "tool", "tool_call_id": tc.id, "content": result}
    ]

    r2 = client.chat.completions.create(
        model=TOOL_MODEL, messages=messages, tools=tools
    )
    print(f"\nFinal answer: {r2.choices[0].message.content}")
    print("\n→ Full cycle: user → model requests tool → we run it → model answers")

Tool result: {"temp": "22\u00b0C", "condition": "partly cloudy"}

Final answer: thought
<channel|>The current weather in Tokyo is 22°C and partly cloudy.

→ Full cycle: user → model requests tool → we run it → model answers
